## 第 4 课：二维 Grid 与 Stride

计算目标：

In [ ]:
output[m, n] = x[m, n] + b[n]

其中：

In [ ]:
x.shape = (M, N)
b.shape = (N,)

例如：

In [ ]:
x = [[1, 2, 3],
     [4, 5, 6]]

b = [10, 20, 30]

output = [[11, 22, 33],
          [14, 25, 36]]

Bias 在每一行重复使用，这就是 broadcasting。

### 1. 二维 Grid

这次一个 program 由两个 ID 确定：

In [ ]:
pid_m = tl.program_id(axis=0)  # 第几行
pid_n = tl.program_id(axis=1)  # 这一行的第几个列块

Grid 是：

In [ ]:
grid = (M, triton.cdiv(N, BLOCK_SIZE_N))

假设：

In [ ]:
M = 2
N = 260
BLOCK_SIZE_N = 128

那么：

In [ ]:
grid = (2, 3)

一共启动 `2 × 3 = 6` 个 program：

In [ ]:
(0, 0) → 第 0 行，第   0～127 列
(0, 1) → 第 0 行，第 128～255 列
(0, 2) → 第 0 行，第 256～259 列

(1, 0) → 第 1 行，第   0～127 列
(1, 1) → 第 1 行，第 128～255 列
(1, 2) → 第 1 行，第 256～259 列

### 2. Stride 是什么？

矩阵元素 `x[m, n]` 的内存偏移量是：

In [ ]:
m * stride_xm + n * stride_xn

对于连续的 `(M, N)` 矩阵：

In [ ]:
stride_xm = N
stride_xn = 1

因此：

In [ ]:
x[2, 5]

对应：

In [ ]:
内存偏移 = 2 × N + 5

### 3. 当前 program 的列下标

In [ ]:
offs_n = (
    pid_n * BLOCK_SIZE_N
    + tl.arange(0, BLOCK_SIZE_N)
)

对应 mask：

In [ ]:
mask = offs_n < N

矩阵元素的地址偏移：

In [ ]:
x_offsets = pid_m * stride_xm + offs_n * stride_xn

Bias 是一维的，不需要行偏移：

In [ ]:
b_ptr + offs_n

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def bias_add_kernel(
    x_ptr,
    b_ptr,
    output_ptr,
    M,
    N,
    stride_xm,
    stride_xn,
    BLOCK_SIZE_N: tl.constexpr,
):
    # TODO 1：取得行 program ID
    pid_row = tl.program_id(axis=0)
    # TODO 2：取得列块 program ID
    pid_col = tl.program_id(axis=1)
    # TODO 3：生成当前列块的 offs_n
    offs_n = pid_col * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    # TODO 4：生成列方向 mask
    # 注意：mask 是"列下标是否合法"，用 N（逻辑边界），不是 stride_xm（内存布局）
    mask = offs_n < N
    # TODO 5：计算 x 当前行、当前列块的内存 offsets
    offsets = pid_row * stride_xm + offs_n * stride_xn
    # TODO 6：读取 x
    x_block = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    # TODO 7：读取 bias
    # 注意：bias 不需要 pid_m
    b_block = tl.load(b_ptr + offs_n, mask=mask, other=0.0)
    # TODO 8：相加
    out = x_block + b_block
    # TODO 9：写入 output（指针在前，值在后）
    tl.store(output_ptr + offsets, out, mask=mask)


def bias_add(
    x: torch.Tensor,
    b: torch.Tensor,
) -> torch.Tensor:
    # 已知 x 是连续二维 Tensor
    # 已知 b 是长度为 N 的连续一维 Tensor

    # TODO 10：检查维度和形状
    # 注意：x.shape 是 tuple，用 [] 索引；b.shape 是 (N,) 不是整数
    assert x.dim() == 2 and b.dim() == 1
    assert x.shape[1] == b.shape[0]
    # TODO 11：取得 M、N
    M = x.shape[0]
    N = x.shape[1]
    # TODO 12：分配 output
    output = torch.empty_like(x)
    BLOCK_SIZE_N = 128

    # TODO 13：创建二维 grid
    grid = (M, triton.cdiv(N, BLOCK_SIZE_N))
    # TODO 14：启动 kernel
    # stride 使用 x.stride(0)、x.stride(1)
    bias_add_kernel[grid](
        x, b, output, M, N,
        x.stride(0), x.stride(1),
        BLOCK_SIZE_N,
    )
    # TODO 15：返回 output
    return output

同时回答：

1. `M=3, N=300, BLOCK_SIZE_N=128` 时，grid 是多少？总共有几个 program？
(3,3),总共9个
2. `pid_m=2, pid_n=1` 负责哪一行、哪些列？
第三行，第129列到256列
3. 为什么读取 bias 时不需要加 `pid_m * stride`？
bias是一维的，加上这个就多了个偏移了，就错了
把代码和三个答案发给我，我继续审查。